# SuperTrader.AI - Intraday DQN Trading System
## Complete Training Notebook with All Fixes

### Key Features:
- ✅ Proper epsilon decay (linear, per training step)
- ✅ Time-based features for intraday patterns
- ✅ Volatility-scaled rewards (as per paper)
- ✅ Double DQN with Dueling architecture
- ✅ Prioritized experience replay
- ✅ Comprehensive logging and visualization
- ✅ Intraday constraints (auto square-off at 3:15 PM)

---

## 1. Setup and Imports

In [2]:
# Install required packages (uncomment if needed)
# !pip install tensorflow pandas numpy matplotlib scikit-learn joblib ta-lib

In [3]:
import os
import sys
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, Model, optimizers
from collections import deque
import random
import logging
from pathlib import Path
from datetime import datetime, time
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, Tuple, List, Optional
import json
import warnings
warnings.filterwarnings('ignore')

# Seaborn styling
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print('✓ Imports complete')
print(f'TensorFlow version: {tf.__version__}')
print(f'NumPy version: {np.__version__}')
print(f'Pandas version: {pd.__version__}')

✓ Imports complete
TensorFlow version: 2.10.1
NumPy version: 1.26.4
Pandas version: 2.3.3


## 2. GPU Configuration

In [4]:
# GPU setup
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        logical_gpus = tf.config.list_logical_devices('GPU')
        print(f'✓ {len(gpus)} Physical GPU(s), {len(logical_gpus)} Logical GPU(s)')
        print('  GPU memory growth enabled')
    except RuntimeError as e:
        print(f'GPU configuration error: {e}')
else:
    print('⚠ No GPU available - training will run on CPU (slower)')

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)
random.seed(42)
print('✓ Random seeds set for reproducibility')

✓ 1 Physical GPU(s), 1 Logical GPU(s)
  GPU memory growth enabled
✓ Random seeds set for reproducibility


## 3. Logging Setup

In [5]:
# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler('training.log')
    ]
)
logger = logging.getLogger('IntradayDQN')
logger.info('Logging initialized')
print('✓ Logging configured')

2025-11-08 02:43:40,652 - IntradayDQN - INFO - Logging initialized


✓ Logging configured


## 4. Environment Class

In [6]:
class IntradayTradingEnv:
    """
    Intraday trading environment with:
    - Time-of-day features
    - Volatility-scaled rewards
    - Mandatory square-off before 3:15 PM
    - Position size scaling based on session
    """
    
    def __init__(
        self,
        data: pd.DataFrame,
        feature_cols: List[str],
        lookback: int = 30,
        initial_balance: float = 100000.0,
        transaction_cost_bps: float = 2.5,
        lot_size: int = 50,
        volatility_target: float = 0.15,
    ):
        self.data = data
        self.feature_cols = feature_cols
        self.lookback = lookback
        self.initial_balance = initial_balance
        self.transaction_cost_bps = transaction_cost_bps
        self.lot_size = lot_size
        self.volatility_target = volatility_target
        self.total_trading_minutes = 360  # 9:15 AM to 3:15 PM
        
        # State tracking
        self.current_step = 0
        self.current_position = 0
        self.balance = initial_balance
        self.equity_curve = []
        self.trade_log = []
        
        # For volatility calculation
        self.price_history = deque(maxlen=60)
        self.returns_history = deque(maxlen=60)
        
        # Episode statistics
        self.episode_trades = 0
        self.episode_wins = 0
        self.episode_losses = 0
        
        logger.info(f'Environment initialized: {len(data)} data points')
    
    def reset(self) -> np.ndarray:
        """Reset environment for new episode."""
        self.current_step = self.lookback
        self.current_position = 0
        self.balance = self.initial_balance
        self.equity_curve = [self.initial_balance]
        self.trade_log = []
        self.episode_trades = 0
        self.episode_wins = 0
        self.episode_losses = 0
        
        # Initialize history
        self.price_history.clear()
        self.returns_history.clear()
        
        for i in range(max(0, self.current_step - 60), self.current_step):
            if i < len(self.data):
                price = self.data.iloc[i]['close']
                self.price_history.append(price)
                if len(self.price_history) > 1:
                    ret = (self.price_history[-1] - self.price_history[-2]) / self.price_history[-2]
                    self.returns_history.append(ret)
        
        return self._get_state()
    
    def _get_state(self) -> np.ndarray:
        """Get current state with price + time features."""
        # Price features
        start_idx = max(0, self.current_step - self.lookback)
        end_idx = self.current_step
        price_features = self.data.iloc[start_idx:end_idx][self.feature_cols].values
        
        # Pad if necessary
        if len(price_features) < self.lookback:
            padding = np.zeros((self.lookback - len(price_features), len(self.feature_cols)))
            price_features = np.vstack([padding, price_features])
        
        # Time features
        minutes_since_open = self._get_minutes_since_open()
        minutes_to_close = self._get_minutes_to_close()
        
        time_features = np.array([
            minutes_since_open / self.total_trading_minutes,
            minutes_to_close / self.total_trading_minutes,
            self._get_session_indicator(),
            self.current_position,
        ])
        
        # Repeat for all timesteps
        time_features_repeated = np.tile(time_features, (self.lookback, 1))
        
        # Concatenate
        state = np.hstack([price_features, time_features_repeated])
        return state.astype(np.float32)
    
    def _get_minutes_since_open(self) -> float:
        return min(self.current_step, self.total_trading_minutes)
    
    def _get_minutes_to_close(self) -> float:
        return max(0, self.total_trading_minutes - self._get_minutes_since_open())
    
    def _get_session_indicator(self) -> float:
        """0=opening, 1=morning, 2=afternoon, 3=closing"""
        minutes = self._get_minutes_since_open()
        if minutes <= 30:
            return 0.0
        elif minutes <= 150:
            return 1.0
        elif minutes <= 300:
            return 2.0
        else:
            return 3.0
    
    def _calculate_volatility(self) -> float:
        """Calculate exponentially weighted volatility."""
        if len(self.returns_history) < 5:
            return 0.15
        
        returns_array = np.array(list(self.returns_history))
        weights = np.exp(np.linspace(-1, 0, len(returns_array)))
        weights = weights / weights.sum()
        
        weighted_std = np.sqrt(np.sum(weights * returns_array**2))
        annualized_std = weighted_std * np.sqrt(252 * 360)
        
        return max(0.05, min(0.50, annualized_std))
    
    def step(self, action: int) -> Tuple[np.ndarray, float, bool, Dict]:
        """Execute one step. Action: 0=short, 1=flat, 2=long"""
        if self.current_step >= len(self.data) - 1:
            return self._get_state(), 0.0, True, {'reason': 'end_of_data'}
        
        # Time constraint - force flat near close
        minutes_to_close = self._get_minutes_to_close()
        if minutes_to_close < 5:
            action = 1  # Force FLAT
        
        # Map action to position
        target_position = action - 1  # 0->-1, 1->0, 2->1
        
        # Get prices
        current_price = self.data.iloc[self.current_step]['close']
        self.current_step += 1
        next_price = self.data.iloc[self.current_step]['close']
        
        # Update history
        self.price_history.append(next_price)
        if len(self.price_history) > 1:
            ret = (self.price_history[-1] - self.price_history[-2]) / self.price_history[-2]
            self.returns_history.append(ret)
        
        # Calculate return
        price_return = (next_price - current_price) / current_price
        
        # Volatility scaling
        current_volatility = self._calculate_volatility()
        volatility_scalar = self.volatility_target / current_volatility
        
        # Position reward (volatility scaled)
        position_reward = self.current_position * volatility_scalar * price_return
        
        # Transaction cost
        position_change = abs(target_position - self.current_position)
        transaction_cost = 0.0
        
        if position_change > 0:
            transaction_cost = (self.transaction_cost_bps / 10000) * position_change
            self.episode_trades += 1
            
            self.trade_log.append({
                'step': self.current_step,
                'from_position': self.current_position,
                'to_position': target_position,
                'price': next_price,
                'cost': transaction_cost,
                'minutes_to_close': minutes_to_close
            })
        
        # Total reward
        reward = position_reward - transaction_cost
        
        # Update balance
        self.balance += reward * self.initial_balance
        self.equity_curve.append(self.balance)
        
        # Track wins/losses
        if reward > 0:
            self.episode_wins += 1
        elif reward < 0:
            self.episode_losses += 1
        
        self.current_position = target_position
        
        # Check if done
        done = False
        info = {}
        
        if minutes_to_close <= 0 or self.current_step >= len(self.data) - 1:
            done = True
            info['reason'] = 'market_close' if minutes_to_close <= 0 else 'end_of_data'
            
            # Exit penalty if still in position
            if self.current_position != 0:
                exit_penalty = 0.01 * abs(self.current_position)
                reward -= exit_penalty
                self.balance -= exit_penalty * self.initial_balance
                info['exit_penalty'] = exit_penalty
        
        next_state = self._get_state()
        
        info.update({
            'step': self.current_step,
            'position': self.current_position,
            'balance': self.balance,
            'volatility': current_volatility,
            'minutes_to_close': minutes_to_close,
            'trades': self.episode_trades
        })
        
        return next_state, reward, done, info
    
    def get_episode_stats(self) -> Dict:
        """Get episode statistics."""
        equity_array = np.array(self.equity_curve)
        returns = np.diff(equity_array) / equity_array[:-1]
        
        total_return = (self.balance - self.initial_balance) / self.initial_balance
        
        sharpe_ratio = 0.0
        if len(returns) > 1 and np.std(returns) > 0:
            sharpe_ratio = np.mean(returns) / np.std(returns) * np.sqrt(252 * 6.5 / 24)
        
        max_drawdown = 0.0
        peak = equity_array[0]
        for value in equity_array:
            if value > peak:
                peak = value
            drawdown = (peak - value) / peak
            if drawdown > max_drawdown:
                max_drawdown = drawdown
        
        win_rate = self.episode_wins / max(1, self.episode_wins + self.episode_losses)
        
        return {
            'total_return': total_return,
            'total_return_pct': total_return * 100,
            'final_balance': self.balance,
            'sharpe_ratio': sharpe_ratio,
            'max_drawdown': max_drawdown,
            'num_trades': self.episode_trades,
            'wins': self.episode_wins,
            'losses': self.episode_losses,
            'win_rate': win_rate,
            'avg_trade_return': np.mean(returns) if len(returns) > 0 else 0.0
        }

print('✓ Environment class defined')

✓ Environment class defined


## 5. Dueling DQN Network

In [7]:
class DuelingDQN_LSTM(Model):
    """Dueling DQN with LSTM layers."""
    
    def __init__(self, n_features, n_actions=3, lstm1_units=64, lstm2_units=32, dropout=0.1):
        super().__init__()
        
        self.n_features = n_features
        self.n_actions = n_actions
        
        # LSTM layers
        self.lstm1 = layers.LSTM(lstm1_units, return_sequences=True, name='lstm1')
        self.lstm2 = layers.LSTM(lstm2_units, return_sequences=False, name='lstm2')
        self.dropout = layers.Dropout(dropout, name='dropout')
        
        # Value stream
        self.value_fc = layers.Dense(lstm2_units, activation='leaky_relu', name='value_fc')
        self.value_out = layers.Dense(1, name='value_output')
        
        # Advantage stream
        self.advantage_fc = layers.Dense(lstm2_units, activation='leaky_relu', name='advantage_fc')
        self.advantage_out = layers.Dense(n_actions, name='advantage_output')
    
    def call(self, x, training=False):
        x = self.lstm1(x, training=training)
        x = self.lstm2(x, training=training)
        x = self.dropout(x, training=training)
        
        # Value stream
        V = self.value_out(self.value_fc(x))
        
        # Advantage stream
        A = self.advantage_out(self.advantage_fc(x))
        A_mean = tf.reduce_mean(A, axis=1, keepdims=True)
        
        # Combine: Q = V + (A - mean(A))
        Q = V + (A - A_mean)
        
        return Q

print('✓ Dueling DQN network defined')

✓ Dueling DQN network defined


## 6. Prioritized Replay Buffer

In [8]:
class PrioritizedReplayBuffer:
    """Experience replay with prioritized sampling."""
    
    def __init__(self, capacity=5000, alpha=0.6):
        self.capacity = capacity
        self.alpha = alpha
        self.buffer = deque(maxlen=capacity)
        self.priorities = deque(maxlen=capacity)
    
    def add(self, state, action, reward, next_state, done, priority=None):
        max_priority = max(self.priorities) if self.priorities else 1.0
        if priority is None:
            priority = max_priority
        
        self.buffer.append((state, action, reward, next_state, done))
        self.priorities.append(priority)
    
    def sample(self, batch_size, beta=0.4):
        if len(self.buffer) == 0:
            return None, None, None
        
        batch_size = min(batch_size, len(self.buffer))
        
        # Calculate probabilities
        priorities = np.array(self.priorities, dtype=np.float32)
        probabilities = priorities ** self.alpha
        probabilities /= probabilities.sum()
        
        # Sample
        indices = np.random.choice(len(self.buffer), batch_size, p=probabilities, replace=False)
        
        # Importance sampling weights
        weights = (len(self.buffer) * probabilities[indices]) ** (-beta)
        weights /= weights.max()
        
        # Get batch
        batch = [self.buffer[idx] for idx in indices]
        states, actions, rewards, next_states, dones = map(np.array, zip(*batch))
        
        return (states, actions, rewards.astype(np.float32), next_states, dones.astype(np.float32)), indices, weights
    
    def update_priorities(self, indices, priorities):
        for idx, priority in zip(indices, priorities):
            self.priorities[idx] = priority + 1e-6
    
    def __len__(self):
        return len(self.buffer)

print('✓ Prioritized replay buffer defined')

✓ Prioritized replay buffer defined


## 7. DQN Agent (WITH EPSILON FIX)

In [9]:
class IntradayDQNAgent:
    """DQN Agent with FIXED epsilon decay."""
    
    def __init__(
        self,
        state_shape,
        n_actions=3,
        gamma=0.3,
        epsilon_start=1.0,
        epsilon_end=0.01,
        epsilon_decay_steps=10000,
        learning_rate=1e-4,
        tau=0.001,
        use_soft_update=True
    ):
        self.state_shape = state_shape
        self.n_actions = n_actions
        self.gamma = gamma
        
        # EPSILON PARAMETERS - KEY FIX
        self.epsilon = epsilon_start
        self.epsilon_start = epsilon_start
        self.epsilon_end = epsilon_end
        self.epsilon_decay_steps = epsilon_decay_steps
        self.epsilon_decay_rate = (epsilon_start - epsilon_end) / epsilon_decay_steps
        self.training_steps = 0  # Track ALL training steps
        
        self.tau = tau
        self.use_soft_update = use_soft_update
        
        # Build networks
        logger.info(f'Building networks with state shape {state_shape}')
        self.policy_net = DuelingDQN_LSTM(n_features=state_shape[1], n_actions=n_actions)
        self.target_net = DuelingDQN_LSTM(n_features=state_shape[1], n_actions=n_actions)
        
        # Initialize
        dummy_input = np.zeros((1, *state_shape), dtype=np.float32)
        self.policy_net(dummy_input)
        self.target_net(dummy_input)
        self.target_net.set_weights(self.policy_net.get_weights())
        
        self.optimizer = optimizers.Adam(learning_rate=learning_rate)
        
        # Metrics
        self.loss_history = []
        self.epsilon_history = []
        self.q_value_history = []
        
        logger.info('Agent initialized')
        logger.info(f'Epsilon decay: {epsilon_start} -> {epsilon_end} over {epsilon_decay_steps} steps')
    
    def select_action(self, state, mode='train', minutes_to_close=180.0):
        """Select action with epsilon-greedy."""
        # Time constraint
        if minutes_to_close < 15:
            return 1, {'reason': 'time_constraint', 'minutes_to_close': minutes_to_close}
        
        minutes_since_open = 360 - minutes_to_close
        is_opening_range = minutes_since_open <= 30
        
        # Epsilon-greedy
        if mode == 'train' and np.random.rand() < self.epsilon:
            action = np.random.randint(0, self.n_actions)
            info = {'mode': 'explore', 'epsilon': self.epsilon, 'is_opening_range': is_opening_range}
        else:
            # Exploit
            state_tensor = tf.convert_to_tensor(state[np.newaxis], dtype=tf.float32)
            q_values = self.policy_net(state_tensor, training=False)
            q_values = q_values.numpy()[0]
            action = int(np.argmax(q_values))
            
            info = {
                'mode': 'exploit',
                'q_values': q_values.tolist(),
                'max_q': float(np.max(q_values)),
                'epsilon': self.epsilon,
                'is_opening_range': is_opening_range
            }
            
            self.q_value_history.append(float(np.max(q_values)))
        
        # Opening range: reduce aggressiveness
        if is_opening_range and action != 1:
            if np.random.rand() < 0.5:
                action = 1
                info['opening_range_override'] = True
        
        return action, info
    
    def train_step(self, batch, indices, weights):
        """Single training step with Double DQN."""
        states, actions, rewards, next_states, dones = batch
        
        # Convert to tensors
        states_tensor = tf.convert_to_tensor(states, dtype=tf.float32)
        next_states_tensor = tf.convert_to_tensor(next_states, dtype=tf.float32)
        rewards_tensor = tf.convert_to_tensor(rewards, dtype=tf.float32)
        dones_tensor = tf.convert_to_tensor(dones, dtype=tf.float32)
        actions_tensor = tf.convert_to_tensor(actions, dtype=tf.int32)
        weights_tensor = tf.convert_to_tensor(weights, dtype=tf.float32)
        
        with tf.GradientTape() as tape:
            # Current Q-values
            q_values = self.policy_net(states_tensor, training=True)
            q_values = tf.gather_nd(q_values, tf.stack([tf.range(len(actions)), actions_tensor], axis=1))
            
            # Double DQN: policy selects, target evaluates
            next_q_policy = self.policy_net(next_states_tensor, training=False)
            next_actions = tf.argmax(next_q_policy, axis=1, output_type=tf.int32)
            
            next_q_target = self.target_net(next_states_tensor, training=False)
            next_q_values = tf.gather_nd(next_q_target, tf.stack([tf.range(len(next_actions)), next_actions], axis=1))
            
            # Targets
            targets = rewards_tensor + self.gamma * next_q_values * (1 - dones_tensor)
            
            # TD-errors
            td_errors = tf.abs(q_values - targets)
            
            # Weighted loss
            loss = tf.reduce_mean(weights_tensor * tf.square(q_values - targets))
        
        # Backpropagation with gradient clipping
        gradients = tape.gradient(loss, self.policy_net.trainable_variables)
        gradients, _ = tf.clip_by_global_norm(gradients, 1.0)
        self.optimizer.apply_gradients(zip(gradients, self.policy_net.trainable_variables))
        
        # KEY FIX: Update epsilon EVERY training step
        self.training_steps += 1
        if self.epsilon > self.epsilon_end:
            self.epsilon = max(
                self.epsilon_end,
                self.epsilon_start - self.epsilon_decay_rate * self.training_steps
            )
        
        # Track metrics
        self.loss_history.append(float(loss.numpy()))
        self.epsilon_history.append(self.epsilon)
        
        return float(loss.numpy()), td_errors.numpy()
    
    def update_target_network(self, soft_update=None):
        """Update target network."""
        if soft_update is None:
            soft_update = self.use_soft_update
        
        if soft_update:
            # Soft update
            for target_var, policy_var in zip(self.target_net.trainable_variables, 
                                               self.policy_net.trainable_variables):
                target_var.assign(self.tau * policy_var + (1 - self.tau) * target_var)
        else:
            # Hard update
            self.target_net.set_weights(self.policy_net.get_weights())
    
    def get_metrics(self):
        """Get training metrics."""
        recent_window = 100
        return {
            'epsilon': self.epsilon,
            'training_steps': self.training_steps,
            'avg_loss': np.mean(self.loss_history[-recent_window:]) if self.loss_history else 0.0,
            'avg_q_value': np.mean(self.q_value_history[-recent_window:]) if self.q_value_history else 0.0,
            'total_updates': len(self.loss_history)
        }
    
    def save_weights(self, filepath):
        self.policy_net.save_weights(filepath)
        logger.info(f'Saved weights to {filepath}')
    
    def load_weights(self, filepath):
        self.policy_net.load_weights(filepath)
        self.target_net.set_weights(self.policy_net.get_weights())
        logger.info(f'Loaded weights from {filepath}')

print('✓ DQN Agent defined (with epsilon fix)')

✓ DQN Agent defined (with epsilon fix)


## 8. Training Loop

In [10]:
def train_intraday_dqn(
    env,
    agent,
    buffer,
    num_episodes=150,
    batch_size=64,
    target_update_freq=1000,
    save_freq=10,
    save_dir='./models',
    log_freq=1
):
    """Main training loop."""
    logger.info('='*80)
    logger.info('STARTING TRAINING')
    logger.info('='*80)
    logger.info(f'Episodes: {num_episodes}')
    logger.info(f'Batch size: {batch_size}')
    logger.info(f'Target update frequency: {target_update_freq}')
    
    # Create save directory
    save_path = Path(save_dir)
    save_path.mkdir(parents=True, exist_ok=True)
    
    # Training history
    episode_rewards = []
    episode_sharpe = []
    episode_trades = []
    episode_win_rates = []
    
    # Beta annealing for importance sampling
    beta_start = 0.4
    beta_end = 1.0
    beta = beta_start
    
    for episode in range(1, num_episodes + 1):
        # Reset
        state = env.reset()
        done = False
        episode_reward = 0.0
        step = 0
        episode_losses = []
        
        if episode % log_freq == 0:
            logger.info(f'\n{"="*80}')
            logger.info(f'Episode {episode}/{num_episodes}')
            logger.info('='*80)
        
        # Episode loop
        while not done:
            # Select action
            minutes_to_close = env._get_minutes_to_close()
            action, action_info = agent.select_action(state, mode='train', minutes_to_close=minutes_to_close)
            
            # Take step
            next_state, reward, done, step_info = env.step(action)
            
            # Add to buffer
            buffer.add(state, action, reward, next_state, done)
            
            episode_reward += reward
            state = next_state
            step += 1
            
            # Training
            if len(buffer) >= batch_size:
                # Anneal beta
                beta = min(beta_end, beta_start + (beta_end - beta_start) * agent.training_steps / 50000)
                
                # Sample and train
                batch_data, indices, weights = buffer.sample(batch_size, beta=beta)
                if batch_data is not None:
                    loss, td_errors = agent.train_step(batch_data, indices, weights)
                    episode_losses.append(loss)
                    buffer.update_priorities(indices, td_errors)
                
                # Update target network
                if agent.training_steps % target_update_freq == 0:
                    agent.update_target_network(soft_update=True)
        
        # Episode complete
        episode_stats = env.get_episode_stats()
        episode_rewards.append(episode_reward)
        episode_sharpe.append(episode_stats['sharpe_ratio'])
        episode_trades.append(episode_stats['num_trades'])
        episode_win_rates.append(episode_stats['win_rate'])
        
        # Logging
        if episode % log_freq == 0:
            agent_metrics = agent.get_metrics()
            
            logger.info(f'\nEPISODE {episode} SUMMARY')
            logger.info('='*80)
            logger.info(f"Total Return: {episode_stats['total_return_pct']:.2f}%")
            logger.info(f"Final Balance: ${episode_stats['final_balance']:.2f}")
            logger.info(f"Sharpe Ratio: {episode_stats['sharpe_ratio']:.3f}")
            logger.info(f"Max Drawdown: {episode_stats['max_drawdown']*100:.2f}%")
            logger.info(f"Trades: {episode_stats['num_trades']} (Wins: {episode_stats['wins']}, Losses: {episode_stats['losses']})")
            logger.info(f"Win Rate: {episode_stats['win_rate']*100:.1f}%")
            logger.info(f"\nAgent Metrics:")
            logger.info(f"  Epsilon: {agent_metrics['epsilon']:.4f}")
            logger.info(f"  Training Steps: {agent_metrics['training_steps']}")
            logger.info(f"  Avg Loss: {agent_metrics['avg_loss']:.6f}")
            logger.info(f"  Buffer Size: {len(buffer)}")
            
            if len(episode_rewards) >= 10:
                logger.info(f"\nRolling Averages (last 10):")
                logger.info(f"  Avg Return: {np.mean(episode_rewards[-10:]):.4f}")
                logger.info(f"  Avg Sharpe: {np.mean(episode_sharpe[-10:]):.3f}")
                logger.info(f"  Avg Trades: {np.mean(episode_trades[-10:]):.1f}")
        
        # Save checkpoint
        if episode % save_freq == 0:
            checkpoint_path = save_path / f'checkpoint_episode_{episode}.weights.h5'
            agent.save_weights(str(checkpoint_path))
            
            # Save history
            history = {
                'episode_rewards': episode_rewards,
                'episode_sharpe': episode_sharpe,
                'episode_trades': episode_trades,
                'episode_win_rates': episode_win_rates,
                'epsilon_history': agent.epsilon_history,
                'loss_history': agent.loss_history
            }
            history_path = save_path / f'training_history_episode_{episode}.json'
            with open(history_path, 'w') as f:
                json.dump({k: [float(v) for v in vals] for k, vals in history.items()}, f, indent=2)
            
            logger.info(f'\n✓ Checkpoint saved: {checkpoint_path}')
    
    logger.info('\n' + '='*80)
    logger.info('TRAINING COMPLETE')
    logger.info('='*80)
    
    # Final save
    final_path = save_path / 'final_model.weights.h5'
    agent.save_weights(str(final_path))
    logger.info(f'Final model saved: {final_path}')
    
    return {
        'episode_rewards': episode_rewards,
        'episode_sharpe': episode_sharpe,
        'episode_trades': episode_trades,
        'episode_win_rates': episode_win_rates,
        'agent': agent
    }

print('✓ Training function defined')

✓ Training function defined


In [52]:
# ========================================
# IMPROVED TRAINING WITH CHECKPOINT MANAGEMENT
# ========================================

def train_intraday_dqn_v2(
    env,
    agent,
    buffer,
    num_episodes=150,
    batch_size=64,
    target_update_freq=1000,
    save_freq=10,
    save_dir='./models',
    log_freq=1,
    resume_from_checkpoint=None,
    experiment_name=None  # NEW: Unique experiment identifier
):
    """
    Enhanced training loop with checkpoint management and resumption.
    
    Features:
    - Timestamped experiment folders to avoid checkpoint conflicts
    - Resume training from specific checkpoints
    - Clear separation between different training runs
    - Comprehensive state saving/loading
    """
    from datetime import datetime
    import shutil
    
    logger.info('='*80)
    logger.info('STARTING ENHANCED TRAINING WITH CHECKPOINT MANAGEMENT')
    logger.info('='*80)
    
    # Create timestamped experiment folder
    if experiment_name is None:
        experiment_name = f"training_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    
    save_path = Path(save_dir) / experiment_name
    save_path.mkdir(parents=True, exist_ok=True)
    
    logger.info(f'Experiment: {experiment_name}')
    logger.info(f'Save directory: {save_path}')
    logger.info(f'Episodes: {num_episodes}')
    logger.info(f'Batch size: {batch_size}')
    logger.info(f'Target update frequency: {target_update_freq}')
    
    # Initialize training state
    start_episode = 1
    episode_rewards = []
    episode_sharpe = []
    episode_trades = []
    episode_win_rates = []
    
    # Resume from checkpoint if specified
    if resume_from_checkpoint:
        checkpoint_info = load_checkpoint(save_path, resume_from_checkpoint, agent, buffer)
        if checkpoint_info:
            start_episode = checkpoint_info['next_episode']
            episode_rewards = checkpoint_info['episode_rewards']
            episode_sharpe = checkpoint_info['episode_sharpe']
            episode_trades = checkpoint_info['episode_trades']
            episode_win_rates = checkpoint_info['episode_win_rates']
            
            logger.info(f'✅ Resumed from episode {resume_from_checkpoint}')
            logger.info(f'   Next episode: {start_episode}')
            logger.info(f'   Training steps: {agent.training_steps}')
            logger.info(f'   Current epsilon: {agent.epsilon:.4f}')
        else:
            logger.warning(f'❌ Could not resume from checkpoint {resume_from_checkpoint}')
            logger.info('   Starting fresh training')
    
    # Clean up conflicting checkpoints
    if not resume_from_checkpoint:
        clean_future_checkpoints(save_path, start_episode, save_freq)
    
    # Beta annealing for importance sampling
    beta_start = 0.4
    beta_end = 1.0
    beta = beta_start
    
    # Save experiment metadata
    save_experiment_metadata(save_path, {
        'experiment_name': experiment_name,
        'start_time': datetime.now().isoformat(),
        'start_episode': start_episode,
        'total_episodes': num_episodes,
        'batch_size': batch_size,
        'target_update_freq': target_update_freq,
        'save_freq': save_freq,
        'agent_config': {
            'gamma': agent.gamma,
            'epsilon_start': agent.epsilon_start,
            'epsilon_end': agent.epsilon_end,
            'epsilon_decay_steps': agent.epsilon_decay_steps,
            'learning_rate': float(agent.optimizer.learning_rate.numpy()),
        },
        'resumed_from': resume_from_checkpoint
    })
    
    for episode in range(start_episode, num_episodes + 1):
        # Reset
        state = env.reset()
        done = False
        episode_reward = 0.0
        step = 0
        episode_losses = []
        
        if episode % log_freq == 0:
            logger.info(f'\n{"="*80}')
            logger.info(f'Episode {episode}/{num_episodes} (Experiment: {experiment_name})')
            logger.info('='*80)
        
        # Episode loop
        while not done:
            # Select action
            minutes_to_close = env._get_minutes_to_close()
            action, action_info = agent.select_action(state, mode='train', minutes_to_close=minutes_to_close)
            
            # Take step
            next_state, reward, done, step_info = env.step(action)
            
            # Add to buffer
            buffer.add(state, action, reward, next_state, done)
            
            episode_reward += reward
            state = next_state
            step += 1
            
            # Training
            if len(buffer) >= batch_size:
                # Anneal beta
                beta = min(beta_end, beta_start + (beta_end - beta_start) * agent.training_steps / 50000)
                
                # Sample and train
                batch_data, indices, weights = buffer.sample(batch_size, beta=beta)
                if batch_data is not None:
                    loss, td_errors = agent.train_step(batch_data, indices, weights)
                    episode_losses.append(loss)
                    buffer.update_priorities(indices, td_errors)
                
                # Update target network
                if agent.training_steps % target_update_freq == 0:
                    agent.update_target_network(soft_update=True)
        
        # Episode complete
        episode_stats = env.get_episode_stats()
        episode_rewards.append(episode_reward)
        episode_sharpe.append(episode_stats['sharpe_ratio'])
        episode_trades.append(episode_stats['num_trades'])
        episode_win_rates.append(episode_stats['win_rate'])
        
        # Logging
        if episode % log_freq == 0:
            agent_metrics = agent.get_metrics()
            
            logger.info(f'\nEPISODE {episode} SUMMARY')
            logger.info('='*80)
            logger.info(f"Total Return: {episode_stats['total_return_pct']:.2f}%")
            logger.info(f"Final Balance: ${episode_stats['final_balance']:.2f}")
            logger.info(f"Sharpe Ratio: {episode_stats['sharpe_ratio']:.3f}")
            logger.info(f"Max Drawdown: {episode_stats['max_drawdown']*100:.2f}%")
            logger.info(f"Trades: {episode_stats['num_trades']} (Wins: {episode_stats['wins']}, Losses: {episode_stats['losses']})")
            logger.info(f"Win Rate: {episode_stats['win_rate']*100:.1f}%")
            logger.info(f"\nAgent Metrics:")
            logger.info(f"  Epsilon: {agent_metrics['epsilon']:.4f}")
            logger.info(f"  Training Steps: {agent_metrics['training_steps']}")
            logger.info(f"  Avg Loss: {agent_metrics['avg_loss']:.6f}")
            logger.info(f"  Buffer Size: {len(buffer)}")
            
            if len(episode_rewards) >= 10:
                logger.info(f"\nRolling Averages (last 10):")
                logger.info(f"  Avg Return: {np.mean(episode_rewards[-10:]):.4f}")
                logger.info(f"  Avg Sharpe: {np.mean(episode_sharpe[-10:]):.3f}")
                logger.info(f"  Avg Trades: {np.mean(episode_trades[-10:]):.1f}")
        
        # Save checkpoint
        if episode % save_freq == 0:
            save_checkpoint(save_path, episode, agent, buffer, {
                'episode_rewards': episode_rewards,
                'episode_sharpe': episode_sharpe,
                'episode_trades': episode_trades,
                'episode_win_rates': episode_win_rates,
            })
    
    logger.info('\n' + '='*80)
    logger.info('TRAINING COMPLETE')
    logger.info('='*80)
    
    # Final save
    final_path = save_path / 'final_model.weights.h5'
    agent.save_weights(str(final_path))
    
    # Save final training history
    final_history = {
        'episode_rewards': episode_rewards,
        'episode_sharpe': episode_sharpe,
        'episode_trades': episode_trades,
        'episode_win_rates': episode_win_rates,
        'epsilon_history': agent.epsilon_history,
        'loss_history': agent.loss_history
    }
    
    history_path = save_path / 'final_training_history.json'
    with open(history_path, 'w') as f:
        json.dump({k: [float(v) for v in vals] for k, vals in final_history.items()}, f, indent=2)
    
    logger.info(f'Final model saved: {final_path}')
    logger.info(f'Final history saved: {history_path}')
    logger.info(f'Experiment folder: {save_path}')
    
    return {
        'episode_rewards': episode_rewards,
        'episode_sharpe': episode_sharpe,
        'episode_trades': episode_trades,
        'episode_win_rates': episode_win_rates,
        'agent': agent,
        'experiment_name': experiment_name,
        'save_path': save_path
    }

print('✅ Enhanced training function with checkpoint management defined')

✅ Enhanced training function with checkpoint management defined


In [25]:
# ========================================
# CHECKPOINT MANAGEMENT UTILITIES
# ========================================

def save_checkpoint(save_path, episode, agent, buffer, training_history):
    """Save comprehensive checkpoint with all training state."""
    checkpoint_dir = save_path / f'checkpoint_episode_{episode}'
    checkpoint_dir.mkdir(exist_ok=True)
    
    # 1. Save model weights
    weights_path = checkpoint_dir / 'model.weights.h5'
    agent.save_weights(str(weights_path))
    
    # 2. Save agent state
    agent_state = {
        'epsilon': agent.epsilon,
        'training_steps': agent.training_steps,
        'epsilon_history': agent.epsilon_history,
        'loss_history': agent.loss_history,
        'q_value_history': agent.q_value_history
    }
    agent_state_path = checkpoint_dir / 'agent_state.json'
    with open(agent_state_path, 'w') as f:
        json.dump({k: [float(v) if isinstance(v, (list, np.ndarray)) else float(v) 
                      for v in (vals if isinstance(vals, (list, np.ndarray)) else [vals])] 
                  for k, vals in agent_state.items()}, f, indent=2)
    
    # 3. Save buffer (optional - can be memory intensive)
    # For now, we'll save buffer metadata only
    buffer_metadata = {
        'size': len(buffer),
        'capacity': buffer.capacity,
        'alpha': buffer.alpha
    }
    buffer_path = checkpoint_dir / 'buffer_metadata.json'
    with open(buffer_path, 'w') as f:
        json.dump(buffer_metadata, f, indent=2)
    
    # 4. Save training history
    history_path = checkpoint_dir / 'training_history.json'
    with open(history_path, 'w') as f:
        json.dump({k: [float(v) for v in vals] for k, vals in training_history.items()}, f, indent=2)
    
    # 5. Save checkpoint metadata
    checkpoint_metadata = {
        'episode': episode,
        'timestamp': datetime.now().isoformat(),
        'training_steps': agent.training_steps,
        'epsilon': agent.epsilon,
        'buffer_size': len(buffer)
    }
    metadata_path = checkpoint_dir / 'metadata.json'
    with open(metadata_path, 'w') as f:
        json.dump(checkpoint_metadata, f, indent=2)
    
    logger.info(f'✅ Checkpoint saved: {checkpoint_dir}')
    return checkpoint_dir

def load_checkpoint(save_path, episode, agent, buffer):
    """Load comprehensive checkpoint and restore training state."""
    checkpoint_dir = save_path / f'checkpoint_episode_{episode}'
    
    if not checkpoint_dir.exists():
        logger.error(f'❌ Checkpoint not found: {checkpoint_dir}')
        return None
    
    try:
        # 1. Load model weights
        weights_path = checkpoint_dir / 'model.weights.h5'
        if weights_path.exists():
            agent.load_weights(str(weights_path))
            logger.info(f'✅ Loaded model weights from {weights_path}')
        
        # 2. Load agent state
        agent_state_path = checkpoint_dir / 'agent_state.json'
        if agent_state_path.exists():
            with open(agent_state_path, 'r') as f:
                agent_state = json.load(f)
            
            agent.epsilon = agent_state['epsilon'][0] if isinstance(agent_state['epsilon'], list) else agent_state['epsilon']
            agent.training_steps = int(agent_state['training_steps'][0] if isinstance(agent_state['training_steps'], list) else agent_state['training_steps'])
            agent.epsilon_history = agent_state['epsilon_history']
            agent.loss_history = agent_state['loss_history']
            agent.q_value_history = agent_state.get('q_value_history', [])
            
            logger.info(f'✅ Loaded agent state: epsilon={agent.epsilon:.4f}, steps={agent.training_steps}')
        
        # 3. Load training history
        history_path = checkpoint_dir / 'training_history.json'
        training_history = {}
        if history_path.exists():
            with open(history_path, 'r') as f:
                training_history = json.load(f)
            logger.info(f'✅ Loaded training history: {len(training_history.get("episode_rewards", []))} episodes')
        
        # 4. Load metadata
        metadata_path = checkpoint_dir / 'metadata.json'
        if metadata_path.exists():
            with open(metadata_path, 'r') as f:
                metadata = json.load(f)
            logger.info(f'✅ Checkpoint from {metadata["timestamp"]}')
        
        return {
            'next_episode': episode + 1,
            'episode_rewards': training_history.get('episode_rewards', []),
            'episode_sharpe': training_history.get('episode_sharpe', []),
            'episode_trades': training_history.get('episode_trades', []),
            'episode_win_rates': training_history.get('episode_win_rates', [])
        }
        
    except Exception as e:
        logger.error(f'❌ Error loading checkpoint: {e}')
        return None

def clean_future_checkpoints(save_path, start_episode, save_freq):
    """Remove checkpoints from future episodes to avoid confusion."""
    import shutil
    
    if start_episode == 1:  # Fresh training
        # Remove all existing checkpoints
        existing_checkpoints = list(save_path.glob('checkpoint_episode_*'))
        if existing_checkpoints:
            logger.info(f'🧹 Cleaning {len(existing_checkpoints)} existing checkpoints for fresh training')
            for checkpoint_dir in existing_checkpoints:
                shutil.rmtree(checkpoint_dir, ignore_errors=True)
            logger.info('✅ Cleanup complete')
    else:  # Resumed training
        # Remove checkpoints beyond current progress
        future_episodes = range(start_episode, 1000, save_freq)  # Conservative upper bound
        for episode in future_episodes:
            checkpoint_dir = save_path / f'checkpoint_episode_{episode}'
            if checkpoint_dir.exists():
                logger.info(f'🧹 Removing future checkpoint: {checkpoint_dir}')
                shutil.rmtree(checkpoint_dir, ignore_errors=True)

def list_available_checkpoints(save_path):
    """List all available checkpoints in the experiment directory."""
    if not save_path.exists():
        print(f"❌ Experiment directory not found: {save_path}")
        return []
    
    checkpoints = []
    for checkpoint_dir in sorted(save_path.glob('checkpoint_episode_*')):
        episode = int(checkpoint_dir.name.split('_')[-1])
        metadata_path = checkpoint_dir / 'metadata.json'
        
        if metadata_path.exists():
            with open(metadata_path, 'r') as f:
                metadata = json.load(f)
            checkpoints.append({
                'episode': episode,
                'timestamp': metadata['timestamp'],
                'training_steps': metadata['training_steps'],
                'epsilon': metadata['epsilon']
            })
        else:
            checkpoints.append({
                'episode': episode,
                'timestamp': 'Unknown',
                'training_steps': 'Unknown',
                'epsilon': 'Unknown'
            })
    
    return checkpoints

def save_experiment_metadata(save_path, metadata):
    """Save experiment-level metadata with proper type conversion."""
    import numpy as np
    
    def convert_to_serializable(obj):
        """Convert TensorFlow and NumPy types to JSON serializable types."""
        # Handle TensorFlow tensors/variables
        if hasattr(obj, 'numpy'):  
            return float(obj.numpy())
        # Handle NumPy types
        elif isinstance(obj, np.floating):
            return float(obj)
        elif isinstance(obj, np.integer):
            return int(obj)
        elif isinstance(obj, np.ndarray):
            return obj.tolist()
        # Handle collections
        elif isinstance(obj, dict):
            return {k: convert_to_serializable(v) for k, v in obj.items()}
        elif isinstance(obj, (list, tuple)):
            return [convert_to_serializable(item) for item in obj]
        # Handle other NumPy scalars
        elif hasattr(obj, 'item'):
            return obj.item()
        # Return as-is for basic Python types
        else:
            return obj
    
    # Convert metadata to serializable format
    serializable_metadata = convert_to_serializable(metadata)
    
    metadata_path = save_path / 'experiment_metadata.json'
    with open(metadata_path, 'w') as f:
        json.dump(serializable_metadata, f, indent=2)
    logger.info(f'✅ Experiment metadata saved: {metadata_path}')

print('✅ Checkpoint management utilities defined')

✅ Checkpoint management utilities defined


## 8.2 Enhanced Training with Checkpoint Management ✨

### 🎯 Problem Solved: Checkpoint Contamination

The original training function had a **major flaw**: 
- **Run 1**: Train 150 episodes → saves checkpoints 10, 20...150
- **Run 2**: Stop at 120 episodes → overwrites checkpoints 10, 20...120
- **Result**: Checkpoints 130, 140, 150 are from **old training** but appear current!

### ✅ New Solution: Timestamped Experiments

Each training run gets its own **unique folder** with complete state management:

```
./models/
├── training_20241108_143022/     ← First run
│   ├── checkpoint_episode_10/
│   │   ├── model.weights.h5
│   │   ├── agent_state.json
│   │   ├── training_history.json
│   │   └── metadata.json
│   ├── checkpoint_episode_20/
│   └── final_model.weights.h5
├── training_20241108_151230/     ← Second run (stopped at 120)
│   ├── checkpoint_episode_10/
│   ├── checkpoint_episode_120/
│   └── experiment_metadata.json
```

### 🔧 Key Features:
- **🏷️ Timestamped experiments** - no confusion between runs
- **📦 Complete state saving** - epsilon, training steps, loss history
- **🔄 Resume capability** - continue from any checkpoint
- **🧹 Auto cleanup** - removes conflicting future checkpoints
- **📊 Rich metadata** - track experiment details

## 9. Visualization Functions

In [26]:
def plot_training_results(history, save_path=None):
    """Plot comprehensive training metrics."""
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    
    # Episode returns
    axes[0, 0].plot(history['episode_rewards'], alpha=0.6, label='Episode Return')
    axes[0, 0].plot(pd.Series(history['episode_rewards']).rolling(10).mean(), label='MA(10)', linewidth=2)
    axes[0, 0].set_title('Episode Returns', fontsize=14, fontweight='bold')
    axes[0, 0].set_xlabel('Episode')
    axes[0, 0].set_ylabel('Return')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # Sharpe ratio
    axes[0, 1].plot(history['episode_sharpe'], alpha=0.6, label='Episode Sharpe')
    axes[0, 1].plot(pd.Series(history['episode_sharpe']).rolling(10).mean(), label='MA(10)', linewidth=2)
    axes[0, 1].set_title('Sharpe Ratio', fontsize=14, fontweight='bold')
    axes[0, 1].set_xlabel('Episode')
    axes[0, 1].set_ylabel('Sharpe')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # Trades
    axes[0, 2].plot(history['episode_trades'], alpha=0.6, label='Trades')
    axes[0, 2].plot(pd.Series(history['episode_trades']).rolling(10).mean(), label='MA(10)', linewidth=2)
    axes[0, 2].set_title('Number of Trades', fontsize=14, fontweight='bold')
    axes[0, 2].set_xlabel('Episode')
    axes[0, 2].set_ylabel('Trades')
    axes[0, 2].legend()
    axes[0, 2].grid(True, alpha=0.3)
    
    # Win rate
    axes[1, 0].plot(history['episode_win_rates'], alpha=0.6, label='Win Rate')
    axes[1, 0].plot(pd.Series(history['episode_win_rates']).rolling(10).mean(), label='MA(10)', linewidth=2)
    axes[1, 0].set_title('Win Rate', fontsize=14, fontweight='bold')
    axes[1, 0].set_xlabel('Episode')
    axes[1, 0].set_ylabel('Win Rate')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    
    # EPSILON DECAY - KEY VISUALIZATION
    axes[1, 1].plot(history['epsilon_history'], label='Epsilon', color='red', alpha=0.7)
    axes[1, 1].set_title('✅ Epsilon Decay (FIXED)', fontsize=14, fontweight='bold')
    axes[1, 1].set_xlabel('Training Step')
    axes[1, 1].set_ylabel('Epsilon')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)
    axes[1, 1].axhline(y=0.01, color='green', linestyle='--', alpha=0.5, label='Target')
    
    # Loss
    axes[1, 2].plot(history['loss_history'], alpha=0.3, label='Loss')
    axes[1, 2].plot(pd.Series(history['loss_history']).rolling(100).mean(), label='MA(100)', linewidth=2)
    axes[1, 2].set_title('Training Loss', fontsize=14, fontweight='bold')
    axes[1, 2].set_xlabel('Training Step')
    axes[1, 2].set_ylabel('Loss')
    axes[1, 2].legend()
    axes[1, 2].grid(True, alpha=0.3)
    axes[1, 2].set_yscale('log')
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        logger.info(f'Plots saved to {save_path}')
    
    plt.show()

print('✓ Visualization functions defined')

✓ Visualization functions defined


## 10. Data Preparation

### Option A: Load Your Real Data

In [48]:
# OPTION A: Load your actual intraday data (ENFORCE real data; no silent synthetic fallback)
from sklearn.preprocessing import StandardScaler
import joblib
from pathlib import Path

# Path to preprocessed/train CSV produced by preprocessing pipeline
data_path = Path('C:/Users/Jugaad/Documents/GitHub/SuperTrader.AI/data/processed_data/train.csv')

# Require processed train CSV to exist. Fail early so notebook does not silently use synthetic data.
if not data_path.exists():
    raise FileNotFoundError(f'Required processed data not found: {data_path}\nPlease run preprocessing to create this file (see data/preprocess_nifty_data.py)')

# Load data (use first column as datetime if present)
print(f'📁 Loading data from: {data_path}')
df = pd.read_csv(data_path, parse_dates=[0], index_col=0)
print(f'✅ Data loaded: {len(df)} samples')
print(f'📅 Date range: {df.index.min()} to {df.index.max()}')

# Show available columns for reference
print(f'📊 Available columns in dataset: {list(df.columns)}')

# Map the available columns to create 12 features that match what the checkpoint expects
# Based on your available columns, select 12 most relevant features
available_features = [
    'norm_close', 'returns', 'log_returns', 'rsi_30', 'macd', 'atr_14',
    'ema_9', 'ema_21', 'ema_distance', 'volatility', 
    'hour', 'minute', 'minutes_since_open', 'minutes_to_close', 'session'
]

# Select exactly 12 features from available ones
feature_cols = [col for col in available_features if col in df.columns][:12]

# If we don't have exactly 12, pad with additional available numeric features
if len(feature_cols) < 12:
    # Add more features from remaining columns
    remaining_numeric = [col for col in df.select_dtypes(include=['number']).columns 
                        if col not in feature_cols and col not in ['open', 'high', 'low', 'close', 'volume']]
    
    # Add until we reach 12 features
    for col in remaining_numeric:
        if len(feature_cols) < 12:
            feature_cols.append(col)

# If still not 12, duplicate some features to maintain checkpoint compatibility
while len(feature_cols) < 12:
    feature_cols.append('volatility')  # Duplicate volatility as safe fallback

# Take exactly 12 features
feature_cols = feature_cols[:12]

print(f'✅ Selected 12-feature set: {feature_cols}')
print(f'✅ Feature count: {len(feature_cols)}')

# Auto-detection override check
detected_feature_cols = None
try:
    if 'experiment_name' in globals() and experiment_name is not None:
        exp_path = Path('./models/intraday_dqn') / experiment_name
        # Try config.json or experiment_metadata.json first
        for cand in ['config.json', 'experiment_metadata.json']:
            cand_path = exp_path / cand
            if cand_path.exists():
                try:
                    meta = json.loads(cand_path.read_text())
                    if 'feature_cols' in meta:
                        detected_feature_cols = meta['feature_cols']
                        print(f'🔍 Auto-detected feature_cols from experiment: {detected_feature_cols}')
                        break
                except Exception:
                    pass
except Exception:
    pass

# Use auto-detected features if they match our available columns and are exactly 12
if detected_feature_cols and len(detected_feature_cols) == 12:
    # Check if all detected features exist in our dataset
    missing_detected = [c for c in detected_feature_cols if c not in df.columns]
    if not missing_detected:
        print(f'✅ Using auto-detected 12 features from experiment')
        feature_cols = list(detected_feature_cols)
    else:
        print(f'⚠️  Auto-detected features missing from dataset: {missing_detected}')
        print(f'✅ Using selected 12 features from available columns')

# Final validation that all features exist
missing = [c for c in feature_cols if c not in df.columns]
if missing:
    available_cols = list(df.columns)
    raise ValueError(f'Missing required feature columns ({len(missing)}): {missing}\nAvailable columns: {available_cols}')

print(f'✅ All 12 features found in dataset')

# Normalize - fit scaler on the training features
print(f'🔄 Normalizing features...')
scaler = StandardScaler()
df[feature_cols] = scaler.fit_transform(df[feature_cols])
# Save scaler for later inference
scaler_path = Path('./models/feature_scaler.pkl')
scaler_path.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(scaler, scaler_path)

print(f'✅ Features normalized and scaler saved to: {scaler_path}')
print(f'✅ Final feature count: {len(feature_cols)} (required for checkpoint compatibility)')
print(f'✅ Features: {feature_cols}')

📁 Loading data from: C:\Users\Jugaad\Documents\GitHub\SuperTrader.AI\data\processed_data\train.csv
✅ Data loaded: 7590 samples
📅 Date range: 2025-04-01 09:25:00 to 2025-08-26 15:00:00
📊 Available columns in dataset: ['exchange_name', 'stock_code', 'date', 'time', 'open', 'high', 'low', 'close', 'volume', 'norm_close', 'returns', 'log_returns', 'rsi_30', 'macd', 'atr_14', 'ema_9', 'ema_21', 'ema_distance', 'volatility', 'hour', 'minute', 'minutes_since_open', 'minutes_to_close', 'session']
✅ Selected 12-feature set: ['norm_close', 'returns', 'log_returns', 'rsi_30', 'macd', 'atr_14', 'ema_9', 'ema_21', 'ema_distance', 'volatility', 'hour', 'minute']
✅ Feature count: 12
✅ All 12 features found in dataset
🔄 Normalizing features...
✅ Features normalized and scaler saved to: models\feature_scaler.pkl
✅ Final feature count: 12 (required for checkpoint compatibility)
✅ Features: ['norm_close', 'returns', 'log_returns', 'rsi_30', 'macd', 'atr_14', 'ema_9', 'ema_21', 'ema_distance', 'volatility

In [49]:
# ========================================
# RECREATE ENVIRONMENT WITH CORRECT 12 FEATURES
# ========================================

print('🔄 Creating environment with 12 features...')

# Recreate environment with the correct 12 features
env = IntradayTradingEnv(
    data=df,
    feature_cols=feature_cols,
    lookback=30,
    initial_balance=100000.0,
    transaction_cost_bps=2.5,  # 2.5 bps realistic for intraday
    lot_size=50,  # NIFTY lot size
    volatility_target=0.15  # 15% target volatility
)

# Test environment
state = env.reset()
print(f'✅ Environment recreated with 12 features')
print(f'  State shape: {state.shape}')
print(f'  Features: {len(feature_cols)} price features + 4 time features = {state.shape[1]} total')
print(f'  Lookback: {env.lookback} timesteps')
print(f'  Trading minutes: {env.total_trading_minutes}')

# Also recreate agent with correct shape
print(f'\n🔄 Creating agent with correct state shape...')

agent = IntradayDQNAgent(
    state_shape=state.shape,
    n_actions=3,
    gamma=0.3,  # Low gamma for intraday (short horizon)
    epsilon_start=1.0,
    epsilon_end=0.01,
    epsilon_decay_steps=10000,  # Decay over 10k training steps
    learning_rate=1e-4,
    tau=0.001,  # Soft update parameter
    use_soft_update=True
)

print(f'✅ Agent recreated')
print(f'  Policy network: {agent.policy_net.count_params():,} parameters')
print(f'  Expected input shape: {agent.state_shape}')
print(f'  Current epsilon: {agent.epsilon:.4f}')

# Recreate buffer
print(f'\n🔄 Creating replay buffer...')

buffer = PrioritizedReplayBuffer(
    capacity=5000,  # ~20-50 episodes worth
    alpha=0.6  # Prioritization exponent
)

print(f'✅ Replay buffer recreated')
print(f'  Capacity: {buffer.capacity}')
print(f'  Current size: {len(buffer)}')

2025-11-08 04:10:27,243 - IntradayDQN - INFO - Environment initialized: 7590 data points
2025-11-08 04:10:27,283 - IntradayDQN - INFO - Building networks with state shape (30, 16)


🔄 Creating environment with 12 features...
✅ Environment recreated with 12 features
  State shape: (30, 16)
  Features: 12 price features + 4 time features = 16 total
  Lookback: 30 timesteps
  Trading minutes: 360

🔄 Creating agent with correct state shape...


2025-11-08 04:10:27,856 - IntradayDQN - INFO - Agent initialized
2025-11-08 04:10:27,857 - IntradayDQN - INFO - Epsilon decay: 1.0 -> 0.01 over 10000 steps


✅ Agent recreated
  Policy network: 35,396 parameters
  Expected input shape: (30, 16)
  Current epsilon: 1.0000

🔄 Creating replay buffer...
✅ Replay buffer recreated
  Capacity: 5000
  Current size: 0


## 🔍 Shape Verification for Checkpoint Compatibility

Before training/resuming, verify that our current setup matches the saved checkpoint expectations.

In [50]:
# ========================================
# SHAPE VERIFICATION FOR CHECKPOINT COMPATIBILITY
# ========================================

print('🔍 VERIFYING SHAPES FOR CHECKPOINT COMPATIBILITY')
print('='*60)

# Verify that environment, agent, and buffer exist
if 'env' not in globals():
    print('❌ Environment not found. Please run the environment creation cells first.')
elif 'agent' not in globals():
    print('❌ Agent not found. Please run the agent creation cells first.')
elif 'buffer' not in globals():
    print('❌ Buffer not found. Please run the buffer creation cells first.')
else:
    # Check state shape
    sample_state = env.reset()
    print(f'📐 Environment state shape: {sample_state.shape}')
    print(f'   - Lookback timesteps: {sample_state.shape[0]}')
    print(f'   - Features per timestep: {sample_state.shape[1]}')

    # Break down features
    price_features = len(feature_cols)
    time_features = 4  # minutes_since_open, minutes_to_close, session, position
    total_features = price_features + time_features

    print(f'\n🧮 Feature breakdown:')
    print(f'   - Price features: {price_features}')
    print(f'   - Time features: {time_features}') 
    print(f'   - Total per timestep: {total_features}')

    # Verify the math
    if sample_state.shape[1] == total_features:
        print(f'✅ Shape verification PASSED: {sample_state.shape[1]} == {total_features}')
    else:
        print(f'❌ Shape verification FAILED: {sample_state.shape[1]} != {total_features}')
        raise ValueError(f'State shape mismatch! Expected {total_features} features per timestep, got {sample_state.shape[1]}')

    # Check agent model input shape
    agent_input_shape = agent.state_shape
    print(f'\n🤖 Agent expected input shape: {agent_input_shape}')
    
    # Verify compatibility
    if sample_state.shape == agent_input_shape:
        print(f'✅ Agent-Environment compatibility PASSED')
    else:
        print(f'❌ Agent-Environment compatibility FAILED')
        print(f'   Environment produces: {sample_state.shape}')
        print(f'   Agent expects: {agent_input_shape}')
        raise ValueError(f'Shape mismatch between environment and agent!')

    # Memory usage check
    state_memory_mb = sample_state.nbytes / (1024 * 1024)
    print(f'\n💾 State memory usage: {state_memory_mb:.2f} MB per state')

    print(f'\n🎯 Expected model input for checkpoint resume:')
    print(f'   - State shape: {sample_state.shape}')
    print(f'   - Feature count: {total_features} (12 price + 4 time)')
    print(f'   - This MUST match the checkpoint from episode 110')

    print(f'\n✅ Shape verification complete - ready for checkpoint resume!')

print('='*60)

🔍 VERIFYING SHAPES FOR CHECKPOINT COMPATIBILITY
📐 Environment state shape: (30, 16)
   - Lookback timesteps: 30
   - Features per timestep: 16

🧮 Feature breakdown:
   - Price features: 12
   - Time features: 4
   - Total per timestep: 16
✅ Shape verification PASSED: 16 == 16

🤖 Agent expected input shape: (30, 16)
✅ Agent-Environment compatibility PASSED

💾 State memory usage: 0.00 MB per state

🎯 Expected model input for checkpoint resume:
   - State shape: (30, 16)
   - Feature count: 16 (12 price + 4 time)
   - This MUST match the checkpoint from episode 110

✅ Shape verification complete - ready for checkpoint resume!


In [51]:
# ========================================
# RESUME TRAINING FROM CHECKPOINT 110 WITH 12 FEATURES
# ========================================

print('🚀 PREPARING TO RESUME FROM CHECKPOINT 110')
print('='*60)

# Ensure we have the right feature count for checkpoint compatibility
if len(feature_cols) != 12:
    raise ValueError(f'Feature count mismatch! Expected 12 features for checkpoint 110, got {len(feature_cols)}')

print(f'✅ Feature count verified: {len(feature_cols)} features')
print(f'📋 Features: {feature_cols}')

# Run migration if not already done
if 'experiment_name' not in globals() or 'latest_episode' not in globals():
    print('🔄 Running checkpoint migration...')
    
    # Execute the migration function (this should have been defined earlier in the notebook)
    if 'migrate_old_checkpoints_and_resume' in globals():
        experiment_name, latest_episode = migrate_old_checkpoints_and_resume()
    else:
        print('❌ Migration function not found. Please run the migration cells first.')
        experiment_name, latest_episode = None, None

if experiment_name and latest_episode:
    print(f'📂 Using experiment: {experiment_name}')
    print(f'📈 Will resume from episode: {latest_episode}')
    print(f'🎯 Target episodes: {TRAINING_CONFIG["num_episodes"]}')
    
    # Verify checkpoint exists
    checkpoint_path = Path(f'./models/intraday_dqn/{experiment_name}/checkpoint_episode_{latest_episode}')
    if checkpoint_path.exists():
        print(f'✅ Checkpoint verified: {checkpoint_path}')
    else:
        print(f'❌ Checkpoint not found: {checkpoint_path}')
        raise FileNotFoundError(f'Expected checkpoint directory not found')
        
    print(f'\n🎯 Ready to resume training with 12 features!')
    print(f'   - Features: {len(feature_cols)}')
    print(f'   - Environment state: {env.reset().shape}')
    print(f'   - Agent input shape: {agent.state_shape}')
    print(f'   - Resume from: Episode {latest_episode}')
    
else:
    print('❌ Migration failed or no checkpoints found')
    print('💡 Will start fresh training instead')

print('='*60)

🚀 PREPARING TO RESUME FROM CHECKPOINT 110
✅ Feature count verified: 12 features
📋 Features: ['norm_close', 'returns', 'log_returns', 'rsi_30', 'macd', 'atr_14', 'ema_9', 'ema_21', 'ema_distance', 'volatility', 'hour', 'minute']
📂 Using experiment: migrated_training_20251108_031129
📈 Will resume from episode: 110
🎯 Target episodes: 150
✅ Checkpoint verified: models\intraday_dqn\migrated_training_20251108_031129\checkpoint_episode_110

🎯 Ready to resume training with 12 features!
   - Features: 12
   - Environment state: (30, 16)
   - Agent input shape: (30, 16)
   - Resume from: Episode 110


In [ ]:
# ========================================
# START TRAINING FROM CHECKPOINT 110 WITH 12 FEATURES
# ========================================

print('🚀 STARTING TRAINING FROM CHECKPOINT 110')
print('='*60)

# Verify we have everything needed
if experiment_name and latest_episode:
    print(f'📂 Experiment: {experiment_name}')
    print(f'📈 Resume from: Episode {latest_episode}')
    print(f'🎯 Continue to: Episode {TRAINING_CONFIG["num_episodes"]}')
    print(f'🧮 Feature count: {len(feature_cols)}')
    print(f'📐 State shape: {env.reset().shape}')
    
    print('\n🏃‍♂️ Starting enhanced training with checkpoint resume...')
    
    # Resume training with dynamic values
    history = train_intraday_dqn_v2(
        env=env,
        agent=agent,
        buffer=buffer,
        experiment_name=experiment_name,           # Use migrated experiment
        resume_from_checkpoint=latest_episode,    # Resume from episode 110
        num_episodes=TRAINING_CONFIG['num_episodes'],
        batch_size=TRAINING_CONFIG['batch_size'],
        target_update_freq=TRAINING_CONFIG['target_update_freq'],
        save_freq=TRAINING_CONFIG['save_freq'],
        save_dir=TRAINING_CONFIG['save_dir'],
        log_freq=TRAINING_CONFIG['log_freq']
    )
    
    print(f'\n🎉 TRAINING COMPLETED!')
    print(f'📊 Experiment saved as: {experiment_name}')
    
else:
    print('❌ Missing experiment_name or latest_episode')
    print('💡 Run the migration cells first')

print('='*60)

2025-11-08 04:15:44,184 - IntradayDQN - INFO - ================================================================================
2025-11-08 04:15:44,186 - IntradayDQN - INFO - STARTING ENHANCED TRAINING WITH CHECKPOINT MANAGEMENT
2025-11-08 04:15:44,186 - IntradayDQN - INFO - ================================================================================
2025-11-08 04:15:44,190 - IntradayDQN - INFO - Experiment: migrated_training_20251108_031129
2025-11-08 04:15:44,191 - IntradayDQN - INFO - Save directory: models\intraday_dqn\migrated_training_20251108_031129
2025-11-08 04:15:44,191 - IntradayDQN - INFO - Episodes: 150
2025-11-08 04:15:44,192 - IntradayDQN - INFO - Batch size: 64
2025-11-08 04:15:44,192 - IntradayDQN - INFO - Target update frequency: 1000
2025-11-08 04:15:44,333 - IntradayDQN - ERROR - ❌ Error loading checkpoint: Cannot assign value to variable ' dueling_dqn_lstm_4/lstm1/lstm_cell_8/kernel:0': Shape mismatch.The variable shape (16, 256), and the assigned value shape (

🚀 STARTING TRAINING FROM CHECKPOINT 110
📂 Experiment: migrated_training_20251108_031129
📈 Resume from: Episode 110
🎯 Continue to: Episode 150
🧮 Feature count: 12
📐 State shape: (30, 16)

🏃‍♂️ Starting enhanced training with checkpoint resume...


2025-11-08 04:15:44,335 - IntradayDQN - WARNING - ❌ Could not resume from checkpoint 110
2025-11-08 04:15:44,335 - IntradayDQN - INFO -    Starting fresh training
2025-11-08 04:15:44,368 - IntradayDQN - INFO - ✅ Experiment metadata saved: models\intraday_dqn\migrated_training_20251108_031129\experiment_metadata.json
2025-11-08 04:15:44,372 - IntradayDQN - INFO - 
2025-11-08 04:15:44,373 - IntradayDQN - INFO - Episode 1/150 (Experiment: migrated_training_20251108_031129)
2025-11-08 04:15:44,374 - IntradayDQN - INFO - ================================================================================
2025-11-08 04:16:09,842 - IntradayDQN - INFO - 
EPISODE 1 SUMMARY
2025-11-08 04:16:09,844 - IntradayDQN - INFO - ================================================================================
2025-11-08 04:16:09,845 - IntradayDQN - INFO - Total Return: -4.87%
2025-11-08 04:16:09,845 - IntradayDQN - INFO - Final Balance: $95126.95
2025-11-08 04:16:09,846 - IntradayDQN - INFO - Sharpe Ratio: -0